### Dynamic examples selection

In [ ]:
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.vectorstores import Qdrant
from langchain.prompts.example_selector import SemanticSimilarityExampleSelector
from langchain.embeddings import HuggingFaceEmbeddings
from qdrant_client import QdrantClient

# 1. Define your example dataset
examples = [
    {"input": "I need help with my billing", "output": "Billing"},
    {"input": "My app keeps crashing", "output": "Technical Support"},
    {"input": "How do I cancel my subscription?", "output": "Cancellation"},
    {"input": "Do you offer student discounts?", "output": "Sales Inquiry"},
    {"input": "I'm having trouble logging in", "output": "Technical Support"},
]

# 2. Define how examples are formatted
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="User: {input}\nCategory: {output}"
)

# 3. Create the selector with FAISS and embeddings
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples=examples,
    # OpenAIEmbeddings can be used as well
    embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
    vectorstore_cls=Qdrant,
    k=2,  # Number of most similar examples to select
    vectorstore_kwargs={
        "client": QdrantClient(":memory:"),
        "collection_name": "example_selector",
        "distance_func": "Cosine",
        "score_threshold": 0.8  # similarity threshold
    }
)

# 4. Create the few-shot prompt template
dynamic_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Classify the category of the following support request.",
    suffix="User: {input}\nCategory:",
    input_variables=["input"]
)

# 5. Generate the prompt for a new user query
user_query = "Can I get a refund?"
final_prompt = dynamic_prompt.format(input=user_query)
print("Generated Prompt:\n" + final_prompt)

In [ ]:
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.vectorstores import FAISS
from langchain.prompts.example_selector import SemanticSimilarityExampleSelector
from langchain.embeddings import HuggingFaceEmbeddings

# 1. Define your example dataset
examples = [
    {"input": "I need help with my billing", "output": "Billing"},
    {"input": "My app keeps crashing", "output": "Technical Support"},
    {"input": "How do I cancel my subscription?", "output": "Cancellation"},
    {"input": "Do you offer student discounts?", "output": "Sales Inquiry"},
    {"input": "I'm having trouble logging in", "output": "Technical Support"},
]

# 2. Define how examples are formatted
example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="User: {input}\nCategory: {output}"
)

# 3. Create the selector with FAISS and embeddings
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples=examples,
    # OpenAIEmbeddings can be used as well
    embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
    vectorstore_cls=FAISS,
    k=2  # Number of most similar examples to select
)

# 4. Create the few-shot prompt template
dynamic_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Classify the category of the following support request.",
    suffix="User: {input}\nCategory:",
    input_variables=["input"]
)

# 5. Generate the prompt for a new user query
user_query = "Can I get a refund?"
final_prompt = dynamic_prompt.format(input=user_query)
print("Generated Prompt:\n" + final_prompt)